# 03 — Extracción de características y entrenamiento

Este cuaderno utiliza las particiones y salidas generadas por `02_preprocessing.ipynb`. Random Forest y SVM reutilizan los 108 atributos (histogramas HSV y estadísticas RGB/HSV); la CNN utiliza las imágenes preprocesadas de 128 × 128. Random Forest y SVM ajustan hiperparámetros con validación cruzada estratificada y consciente de grupos (3 folds) y Macro-F1; la CNN emplea validación, parada temprana y reducción adaptativa de la tasa de aprendizaje.

La prueba (`test`) no se usa para selección ni ajuste. Si falta alguna clase, el resultado se marca como piloto binario/incompleto y no como evaluación final de tres clases. La línea base es `DummyClassifier(strategy="most_frequent")`. Semilla: 42.

In [1]:
from importlib.metadata import version
from pathlib import Path
import json
import sys
import time

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra el cuaderno desde la raíz del repositorio.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.feature_extraction import FEATURE_DIM, load_split_groups
from src.evaluation.metrics import evaluate_predictions
from src.training.train_cnn import train_cnn
from src.training.train_random_forest import train_random_forest
from src.training.train_svm import train_svm
from src.training.group_cv import build_group_cv_splits
from src.utils.config import load_label_info

SEED = 42
PACKAGE_VERSIONS = {
    package: version(package)
    for package in ('numpy', 'pandas', 'scikit-learn', 'tensorflow', 'opencv-python')
}
print('Versiones:', PACKAGE_VERSIONS)
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
RESULTS_DIR = ROOT / 'experiments' / 'results' / 'training'
required_files = [
    PROCESSED_DIR / name
    for name in ('train.csv', 'val.csv', 'test.csv', 'labelinfo.json',
                 'Xtrainml.npy', 'ytrainml.npy', 'Xvalml.npy', 'yvalml.npy')
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan salidas de 02_preprocessing.ipynb: ' + ', '.join(missing_files)
    )

label_info = load_label_info(PROCESSED_DIR)
class_names = label_info['class_names']
label_map = label_info['label_map']
X_train = np.load(PROCESSED_DIR / 'Xtrainml.npy')
y_train = np.load(PROCESSED_DIR / 'ytrainml.npy')
X_val = np.load(PROCESSED_DIR / 'Xvalml.npy')
y_val = np.load(PROCESSED_DIR / 'yvalml.npy')
assert X_train.ndim == X_val.ndim == 2
assert X_train.shape[1] == X_val.shape[1] == FEATURE_DIM == 108
assert len(X_train) == len(y_train) and len(X_val) == len(y_val)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

splits = {
    'train': pd.read_csv(PROCESSED_DIR / 'train.csv'),
    'validation': pd.read_csv(PROCESSED_DIR / 'val.csv'),
    'test': pd.read_csv(PROCESSED_DIR / 'test.csv'),
}
required_manifest_columns = {'quality', 'group_id'}
for split_name, split_df in splits.items():
    missing_columns = required_manifest_columns - set(split_df.columns)
    if missing_columns:
        raise ValueError(f'{split_name} no contiene {sorted(missing_columns)}')
    unknown = set(split_df['quality'].dropna()) - set(class_names)
    if unknown:
        raise ValueError(f'Etiquetas desconocidas en {split_name}: {unknown}')

for left, right in (('train', 'validation'), ('train', 'test'),
                    ('validation', 'test')):
    overlap = set(splits[left]['group_id'].astype(str)) & set(
        splits[right]['group_id'].astype(str)
    )
    if overlap:
        raise ValueError(f'Fuga de grupos entre {left} y {right}: {len(overlap)}')

counts = pd.DataFrame({
    name: frame['quality'].value_counts().reindex(class_names, fill_value=0)
    for name, frame in splits.items()
}).astype(int)
display(counts)
missing_by_split = {
    split_name: counts.index[counts[split_name] == 0].tolist()
    for split_name in counts.columns
}
RESULT_SCOPE = (
    'evaluacion_tres_clases'
    if all(not missing for missing in missing_by_split.values())
    else 'piloto_binario_o_incompleto'
)
if len(np.unique(y_train)) < 2:
    raise ValueError('Se necesitan al menos dos clases en entrenamiento.')
if pd.Series(y_train).value_counts().min() < 3:
    raise ValueError('Cada clase de entrenamiento requiere al menos 3 ejemplos para CV=3.')
if RESULT_SCOPE != 'evaluacion_tres_clases':
    print('ADVERTENCIA: ejecución provisional; clases faltantes por partición:', missing_by_split)
else:
    print('Las tres clases están presentes en train, validation y test.')
print('Alcance de los resultados:', RESULT_SCOPE)
print('Forma de X_train:', X_train.shape, '| características:', FEATURE_DIM)
train_groups = load_split_groups(
    PROCESSED_DIR, 'train', label_info, expected_labels=y_train
)
cv_splits = build_group_cv_splits(y_train, train_groups, n_splits=3, random_state=SEED)
fold_count_rows = []
for fold_number, (train_indices, validation_indices) in enumerate(cv_splits, start=1):
    row = {'fold': fold_number}
    for label, quality in enumerate(class_names):
        row[f'{quality}_train'] = int(np.sum(y_train[train_indices] == label))
        row[f'{quality}_validation'] = int(np.sum(y_train[validation_indices] == label))
    fold_count_rows.append(row)
fold_counts = pd.DataFrame(fold_count_rows)
display(fold_counts)


Versiones: {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.9.1', 'tensorflow': '2.21.0', 'opencv-python': '4.12.0.88'}


,train,validation,test
quality,,,
Good,8459,1813,1813
Regular,0,0,0
Bad,4752,1018,1018


ADVERTENCIA: ejecución provisional; clases faltantes por partición: {'train': ['Regular'], 'validation': ['Regular'], 'test': ['Regular']}
Alcance de los resultados: piloto_binario_o_incompleto
Forma de X_train: (13211, 108) | características: 108


,fold,Good_train,Good_validation,Regular_train,Regular_validation,Bad_train,Bad_validation
0,1,5639,2820,0,0,3168,1584
1,2,5640,2819,0,0,3168,1584
2,3,5639,2820,0,0,3168,1584


## Línea base y ajuste de modelos

El clasificador de clase mayoritaria se ajusta únicamente con `train` y se informa sobre `validation`. RF y SVM usan las búsquedas de `src/models` mediante `GridSearchCV(scoring="f1_macro", cv=3)`. Para la CNN se mantiene la arquitectura existente, Adam, entropía cruzada dispersa, aumento de datos, pesos de clase, `EarlyStopping` y `ReduceLROnPlateau`; sus curvas se guardan para el cuaderno 04. El conjunto de prueba no se consulta durante este paso.

In [2]:
records = {}

baseline = DummyClassifier(strategy='most_frequent', random_state=SEED)
start = time.perf_counter()
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
baseline_seconds = time.perf_counter() - start
baseline_metrics = evaluate_predictions(
    y_val, baseline_pred, class_names, 'Most frequent baseline', 'Validation'
)
records['Baseline'] = {
    'elapsed_seconds': baseline_seconds,
    'validation': baseline_metrics,
}

start = time.perf_counter()
_, rf_metrics = train_random_forest(PROCESSED_DIR, MODELS_DIR)
records['Random Forest'] = {
    'elapsed_seconds': time.perf_counter() - start,
    'validation': rf_metrics,
}

start = time.perf_counter()
_, _, svm_metrics = train_svm(PROCESSED_DIR, MODELS_DIR)
records['SVM'] = {
    'elapsed_seconds': time.perf_counter() - start,
    'validation': svm_metrics,
}

start = time.perf_counter()
_, cnn_metrics = train_cnn(PROCESSED_DIR, MODELS_DIR, epochs=15, batch_size=64)
records['CNN'] = {
    'elapsed_seconds': time.perf_counter() - start,
    'validation': cnn_metrics,
}

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(baseline, MODELS_DIR / 'baseline_most_frequent.pkl')
summary_rows = []
for model_name, detail in records.items():
    validation = detail['validation']
    summary_rows.append({
        'model': model_name,
        'validation_accuracy': validation['accuracy'],
        'validation_macro_f1': validation['macro_f1'],
        'best_cv_macro_f1': validation.get('best_cv_f1_macro', np.nan),
        'elapsed_seconds': detail['elapsed_seconds'],
        'best_params': validation.get('best_params', {}),
    })
training_summary = pd.DataFrame(summary_rows)
training_summary.to_csv(RESULTS_DIR / 'training_summary.csv', index=False)
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'No se puede serializar {type(value).__name__}')

details = {
    'seed': SEED,
    'package_versions': PACKAGE_VERSIONS,
    'result_scope': RESULT_SCOPE,
    'class_names': class_names,
    'split_counts': counts.to_dict(),
    'cv_fold_counts': fold_counts.to_dict(orient='records'),
    'models': records,
}
with (RESULTS_DIR / 'training_details.json').open('w', encoding='utf-8') as file:
    json.dump(details, file, indent=2, default=json_default, allow_nan=True)
display(training_summary)
print('Resumen guardado en:', RESULTS_DIR)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Fitting 3 folds for each of 12 candidates, totalling 36 fits


C:\Users\JuanJ\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Found 13211 images belonging to 3 classes.
Found 2831 images belonging to 3 classes.
Found 2831 images belonging to 3 classes.
Epoch 1/15
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 601ms/step - accuracy: 0.8737 - loss: 0.4320
Epoch 1: val_accuracy improved from None to 0.47828, saving model to C:\Apo3\Proyecto-APO\apo3-mangoMelocoton\models\cnn_best.keras

Epoch 1: finished saving model to C:\Apo3\Proyecto-APO\apo3-mangoMelocoton\models\cnn_best.keras
207/207 ━━━━━━━━━━━━━━━━━━━━ 132s 626ms/step - accuracy: 0.8737 - loss: 0.4320 - val_accuracy: 0.4783 - val_loss: 1.2462 - learning_rate: 0.0010
Epoch 2/15
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9195 - loss: 0.2144
Epoch 2: val_accuracy improved from 0.47828 to 0.83363, saving model to C:\Apo3\Proyecto-APO\apo3-mangoMelocoton\models\cnn_best.keras

Epoch 2: finished saving model to C:\Apo3\Proyecto-APO\apo3-mangoMelocoton\models\cnn_best.keras
207/207 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.9195 - loss: 0.2144 - val_accuracy: 

,model,validation_accuracy,validation_macro_f1,best_cv_macro_f1,elapsed_seconds,best_params
0,Baseline,0.640410,0.260264,NaN,0.004463,{}
1,Random Forest,0.978806,0.651347,0.975385,41.997896,"{'max_depth': None, 'min_samples_split': 2, 'n..."
2,SVM,0.980219,0.652387,0.980091,291.133390,"{'svm__C': 10, 'svm__gamma': 'scale', 'svm__ke..."
3,CNN,0.962557,0.641028,NaN,3458.822172,{}


Resumen guardado en: C:\Apo3\Proyecto-APO\apo3-mangoMelocoton\experiments\results\training


La tabla contiene métricas de validación y tiempos medidos en esta ejecución; `best_cv_macro_f1` y `best_params` aplican a RF/SVM. No se incluyen valores de prueba ni resultados supuestos. Si `result_scope` no es `evaluacion_tres_clases`, el piloto binario/incompleto no satisface la evaluación final y debe describirse como tal.